In [17]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold

from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

In [18]:
df = pd.read_csv('data-week-3-lead.csv')
df.columns = df.columns.str.lower().str.replace(' ' , '_')
categorical_columns  = list(df.dtypes[df.dtypes == 'str'].index)
for c in categorical_columns:
    df[c]=df[c].str.lower().str.replace(' ' , '_')
df.annual_income = df.annual_income.fillna(df['annual_income'].median())
df.lead_score = df.lead_score.fillna(df['lead_score'].mean())
df['lead_source'] = df['lead_source'].fillna(df['lead_source'].mode()[0])
df['industry'] = df['industry'].fillna(df['industry'].mode()[0])
df['employment_status'] = df['employment_status'].fillna(df['employment_status'].mode()[0])
df['location'] = df['location'].fillna(df['location'].mode()[0])

In [19]:
df_full_train , df_test = train_test_split(df , test_size = 0.2 , random_state = 1)

In [20]:
numerical = ['annual_income' , 'number_of_courses_viewed' , 'interaction_count' , 'lead_score']
categorical = [
    'lead_source' , 
    'industry' ,
    'employment_status' ,
    'location',
]

In [21]:
def train(df_train, y_train ,C =1.0):
    dicts = df_train[categorical + numerical].to_dict(orient='records')

    dv = DictVectorizer(sparse=False)
    X_train = dv.fit_transform(dicts)

    model = LogisticRegression(C=C ,max_iter=1000)
    model.fit(X_train, y_train)
    
    return dv, model

In [22]:
def predict(df, dv, model):
    dicts = df[categorical + numerical].to_dict(orient='records')

    X = dv.transform(dicts)
    y_pred = model.predict_proba(X)[:, 1]

    return y_pred

In [23]:
C = 5.0
n_splits = 5

In [24]:
kfold = KFold(n_splits=n_splits, shuffle=True, random_state=1)

scores = []

for train_idx, val_idx in kfold.split(df_full_train):
    df_train = df_full_train.iloc[train_idx]
    df_val = df_full_train.iloc[val_idx]

    y_train = df_train.converted.values
    y_val = df_val.converted.values

    dv, model = train(df_train, y_train, C=C)
    y_pred = predict(df_val, dv, model)

    auc = roc_auc_score(y_val, y_pred)
    scores.append(auc)

print('C=%s %.3f +- %.3f' % (C, np.mean(scores), np.std(scores)))

/usr/local/python/3.14.2/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/usr/local/python/3.14.2/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preproc

C=5.0 0.792 +- 0.012


/usr/local/python/3.14.2/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [25]:
scores

[0.7768833253089392,
 0.7885628244143863,
 0.8000656472042014,
 0.8102594573182809,
 0.7823997519636212]

In [26]:
dv, model = train(df_full_train, df_full_train.converted.values, C=5.0)
y_test = df_test['converted'].values
y_pred = predict(df_test, dv, model)
auc = roc_auc_score(y_test, y_pred)
auc

/usr/local/python/3.14.2/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


0.794384943922796

In [27]:
import pickle

In [28]:
output_file = f'model_C={C}.bin'

In [29]:
with open(output_file, 'wb') as f_out: 
    pickle.dump((dv, model), f_out)

In [30]:
import pickle

In [31]:
input_file = 'model.bin'

In [32]:
with open(input_file, 'rb') as f_in: 
    dv, model = pickle.load(f_in)